# NL → SQL Demo (rule-based, read-only)

This notebook connects to `meta_ads.db`, converts a small set of natural-language questions into SQL using **rules/templates**, executes **SELECT-only** queries safely, and summarizes the outputs.

Notes:
- This is **not** a general NL→SQL system; it supports only the patterns listed below.
- Everything runs locally. No external APIs.


In [1]:
import re
import sqlite3
from pathlib import Path
from textwrap import indent

import pandas as pd

DB_PATH = Path("meta_ads.db")
assert DB_PATH.exists(), f"DB not found at {DB_PATH.resolve()}"

conn = sqlite3.connect(DB_PATH)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [2]:
# Schema hints (from SQLite)
def get_schema_hints(conn: sqlite3.Connection):
    tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
    hints = {}
    for t in tables:
        cols = conn.execute(f"PRAGMA table_info({t})").fetchall()
        hints[t] = [c[1] for c in cols]
    return hints

schema_hints = get_schema_hints(conn)
schema_hints


{'ad_products': ['product_id',
  'product_name',
  'placement',
  'objective',
  'pricing_model',
  'active_from'],
 'ad_transactions': ['transaction_id',
  'event_ts',
  'advertiser_id',
  'product_id',
  'campaign_id',
  'currency',
  'impressions',
  'clicks',
  'conversions',
  'spend_usd',
  'revenue_usd'],
 'advertisers': ['advertiser_id',
  'advertiser_name',
  'industry',
  'hq_country',
  'account_tier',
  'created_at']}

In [3]:
# Read-only SQL guard + executor
BLOCKLIST = {
    "insert", "update", "delete", "drop", "alter", "create",
    "attach", "detach", "pragma", "reindex", "vacuum",
    "replace", "truncate",
}

def _strip_sql_comments(s: str) -> str:
    s = re.sub(r"--.*?$", "", s, flags=re.MULTILINE)
    s = re.sub(r"/\*.*?\*/", "", s, flags=re.DOTALL)
    return s

def is_readonly_sql(sql: str) -> bool:
    sql = _strip_sql_comments(sql).strip()
    if not sql:
        return False
    if ";" in sql:
        return False
    lowered = sql.lower().strip()
    if not lowered.startswith("select"):
        return False
    tokens = set(re.findall(r"[a-zA-Z_]+", lowered))
    if tokens & BLOCKLIST:
        return False
    return True

def run_sql(sql: str) -> pd.DataFrame:
    if not is_readonly_sql(sql):
        raise ValueError("Blocked: only single-statement SELECT queries are allowed")
    return pd.read_sql_query(sql, conn)


## Supported natural-language patterns

- Totals/KPIs: total spend, total revenue, overall roas
- Top-K: top N advertisers/products by revenue/spend/roas
- Breakdowns: revenue/spend/roas by placement/objective/pricing model/industry/tier
- Time series: daily revenue last X days, weekly revenue last X weeks

Optional filters (can be combined):
- last X days
- industry = Ecommerce (any industry in the data)
- tier = SMB | MidMarket | Enterprise
- placement = Feed | Reels | Stories | Search


In [4]:
# Rule-based NL -> SQL
def normalize_nl(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9\s=]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

METRIC_SQL = {
    "spend": "SUM(t.spend_usd)",
    "revenue": "SUM(t.revenue_usd)",
    "roas": "SUM(t.revenue_usd) / NULLIF(SUM(t.spend_usd), 0)",
}

GROUP_DIM = {
    "advertiser": ("a.advertiser_name", "advertiser_name"),
    "advertisers": ("a.advertiser_name", "advertiser_name"),
    "product": ("p.product_name", "product_name"),
    "products": ("p.product_name", "product_name"),
    "placement": ("p.placement", "placement"),
    "objective": ("p.objective", "objective"),
    "pricing model": ("p.pricing_model", "pricing_model"),
    "pricing": ("p.pricing_model", "pricing_model"),
    "industry": ("a.industry", "industry"),
    "tier": ("a.account_tier", "account_tier"),
}

def extract_last_days(nl: str):
    m = re.search(r"last (\d+) days", nl)
    return int(m.group(1)) if m else None

def extract_last_weeks(nl: str):
    m = re.search(r"last (\d+) weeks", nl)
    return int(m.group(1)) if m else None

def extract_top_k(nl: str):
    m = re.search(r"top (\d+)", nl)
    return int(m.group(1)) if m else None

def extract_filter(nl: str, key: str):
    m = re.search(rf"{re.escape(key)}\s*=\s*([a-z0-9]+)", nl)
    return m.group(1) if m else None

def build_where(nl: str):
    clauses = []
    last_days = extract_last_days(nl)
    if last_days is not None:
        clauses.append(f"t.event_ts >= datetime('now','-{last_days} day')")

    industry = extract_filter(nl, "industry")
    if industry is not None:
        clauses.append("LOWER(a.industry) = '" + industry.lower() + "'")

    tier = extract_filter(nl, "tier")
    if tier is not None:
        clauses.append("LOWER(a.account_tier) = '" + tier.lower() + "'")

    placement = extract_filter(nl, "placement")
    if placement is not None:
        clauses.append("LOWER(p.placement) = '" + placement.lower() + "'")

    return ("WHERE " + " AND ".join(clauses)) if clauses else ""

def nl_to_sql(nl: str) -> str:
    raw = nl
    nl = normalize_nl(nl)

    if "total revenue" in nl:
        where = build_where(nl)
        return f"SELECT {METRIC_SQL['revenue']} AS total_revenue_usd FROM ad_transactions t {where}".replace("  ", " ")

    if "total spend" in nl:
        where = build_where(nl)
        return f"SELECT {METRIC_SQL['spend']} AS total_spend_usd FROM ad_transactions t {where}".replace("  ", " ")

    if "overall roas" in nl or nl.strip() == "roas":
        where = build_where(nl)
        return f"SELECT {METRIC_SQL['roas']} AS overall_roas FROM ad_transactions t {where}".replace("  ", " ")

    if nl.startswith("daily revenue"):
        days = extract_last_days(nl) or 30
        return (
            "SELECT date(t.event_ts) AS day, "
            "SUM(t.revenue_usd) AS revenue_usd "
            "FROM ad_transactions t "
            f"WHERE t.event_ts >= datetime('now','-{days} day') "
            "GROUP BY day ORDER BY day"
        )

    if nl.startswith("weekly revenue"):
        weeks = extract_last_weeks(nl) or 12
        return (
            "SELECT strftime('%Y-%W', t.event_ts) AS year_week, "
            "SUM(t.revenue_usd) AS revenue_usd "
            "FROM ad_transactions t "
            f"WHERE t.event_ts >= datetime('now','-{weeks * 7} day') "
            "GROUP BY year_week ORDER BY year_week"
        )

    if nl.startswith("top ") and " by " in nl:
        k = extract_top_k(nl) or 10
        metric = "revenue" if "by revenue" in nl else "spend" if "by spend" in nl else "roas" if "by roas" in nl else None
        if metric is None:
            raise ValueError(f"Unsupported metric in: {raw}")

        if "advertiser" in nl or "advertisers" in nl:
            where = build_where(nl)
            return (
                "SELECT a.advertiser_name, "
                f"{METRIC_SQL[metric]} AS {metric} "
                "FROM ad_transactions t "
                "JOIN advertisers a ON a.advertiser_id = t.advertiser_id "
                "JOIN ad_products p ON p.product_id = t.product_id "
                f"{where} "
                "GROUP BY a.advertiser_name "
                f"ORDER BY {metric} DESC "
                f"LIMIT {k}"
            ).replace("  ", " ")

        if "product" in nl or "products" in nl:
            where = build_where(nl)
            return (
                "SELECT p.product_name, "
                f"{METRIC_SQL[metric]} AS {metric} "
                "FROM ad_transactions t "
                "JOIN advertisers a ON a.advertiser_id = t.advertiser_id "
                "JOIN ad_products p ON p.product_id = t.product_id "
                f"{where} "
                "GROUP BY p.product_name "
                f"ORDER BY {metric} DESC "
                f"LIMIT {k}"
            ).replace("  ", " ")

        raise ValueError(f"Unsupported top-k target in: {raw}")

    m = re.search(r"(revenue|spend|roas) by (placement|objective|pricing model|pricing|industry|tier)", nl)
    if m:
        metric = m.group(1)
        dim_key = m.group(2)
        dim_expr, dim_alias = GROUP_DIM[dim_key]
        where = build_where(nl)
        return (
            f"SELECT {dim_expr} AS {dim_alias}, {METRIC_SQL[metric]} AS {metric} "
            "FROM ad_transactions t "
            "JOIN advertisers a ON a.advertiser_id = t.advertiser_id "
            "JOIN ad_products p ON p.product_id = t.product_id "
            f"{where} "
            f"GROUP BY {dim_alias} "
            f"ORDER BY {metric} DESC"
        ).replace("  ", " ")

    raise ValueError(f"Unsupported NL query (out of scope for rule-based demo): {raw}")


In [5]:
# Query suite (easy -> hard)
nl_queries = [
    "What is total revenue?",
    "What is overall ROAS?",
    "Top 10 advertisers by revenue",
    "Revenue by placement",
    "Daily revenue last 30 days",
    "Top 5 products by ROAS last 60 days",
    "Revenue by industry for tier = Enterprise last 90 days",
    "Top 10 advertisers by revenue for placement = Reels last 45 days",
]

rows = []
for q in nl_queries:
    try:
        sql = nl_to_sql(q)
        ok = is_readonly_sql(sql)
        rows.append({"nl": q, "sql": sql, "readonly_ok": ok})
    except Exception as e:
        rows.append({"nl": q, "sql": f"ERROR: {e}", "readonly_ok": False})

pd.DataFrame(rows)


,nl,sql,readonly_ok
0,What is total revenue?,SELECT SUM(t.revenue_usd) AS total_revenue_usd...,True
1,What is overall ROAS?,SELECT SUM(t.revenue_usd) / NULLIF(SUM(t.spend...,True
2,Top 10 advertisers by revenue,"SELECT a.advertiser_name, SUM(t.revenue_usd) A...",True
3,Revenue by placement,"SELECT p.placement AS placement, SUM(t.revenue...",True
4,Daily revenue last 30 days,"SELECT date(t.event_ts) AS day, SUM(t.revenue_...",True
5,Top 5 products by ROAS last 60 days,"SELECT p.product_name, SUM(t.revenue_usd) / NU...",True
6,Revenue by industry for tier = Enterprise last...,"SELECT a.industry AS industry, SUM(t.revenue_u...",True
7,Top 10 advertisers by revenue for placement = ...,"SELECT a.advertiser_name, SUM(t.revenue_usd) A...",True


In [6]:
# Execute a subset and show outputs
to_run = [
    "What is total revenue?",
    "What is overall ROAS?",
    "Revenue by placement",
    "Top 10 advertisers by revenue",
    "Top 5 products by ROAS last 60 days",
]

results = {}
for q in to_run:
    sql = nl_to_sql(q)
    df_out = run_sql(sql)
    results[q] = df_out
    print("NL:", q)
    print("SQL:\n" + indent(sql, "  "))
    display(df_out.head(10))
    print("-" * 80)


NL: What is total revenue?
SQL:
  SELECT SUM(t.revenue_usd) AS total_revenue_usd FROM ad_transactions t 


,total_revenue_usd
0,3983589.57


--------------------------------------------------------------------------------
NL: What is overall ROAS?
SQL:
  SELECT SUM(t.revenue_usd) / NULLIF(SUM(t.spend_usd), 0) AS overall_roas FROM ad_transactions t 


,overall_roas
0,1.508014


--------------------------------------------------------------------------------
NL: Revenue by placement
SQL:
  SELECT p.placement AS placement, SUM(t.revenue_usd) AS revenue FROM ad_transactions t JOIN advertisers a ON a.advertiser_id = t.advertiser_id JOIN ad_products p ON p.product_id = t.product_id GROUP BY placement ORDER BY revenue DESC


...

--------------------------------------------------------------------------------
NL: Top 10 advertisers by revenue
SQL:
  SELECT a.advertiser_name, SUM(t.revenue_usd) AS revenue FROM ad_transactions t JOIN advertisers a ON a.advertiser_id = t.advertiser_id JOIN ad_products p ON p.product_id = t.product_id GROUP BY a.advertiser_name ORDER BY revenue DESC LIMIT 10


,advertiser_name,revenue
0,Nova Commerce,319615.86
1,Nimbus Finance,291776.18
2,Apex Systems,290167.85
3,Apex Retail,234578.71
4,Orbit Retail,190783.56
5,Nova Finance,177710.00
6,Nova Labs,172790.11
7,Quantum Retail,161386.87
8,Blue Commerce,143019.77
9,Cedar Finance,117707.48


--------------------------------------------------------------------------------
NL: Top 5 products by ROAS last 60 days
SQL:
  SELECT p.product_name, SUM(t.revenue_usd) / NULLIF(SUM(t.spend_usd), 0) AS roas FROM ad_transactions t JOIN advertisers a ON a.advertiser_id = t.advertiser_id JOIN ad_products p ON p.product_id = t.product_id WHERE t.event_ts >= datetime('now','-60 day') GROUP BY p.product_name ORDER BY roas DESC LIMIT 5


,product_name,roas
0,App Install Ads,1.857076
1,Marketplace Ads,1.812753
2,Search Ads,1.780010
3,Lead Gen Ads,1.645541
4,Video In-Stream Ads,1.324945


--------------------------------------------------------------------------------


## Findings summary (auto-generated)

Run the next cell after executing the queries above.


In [7]:
# Summarize findings in natural language
total_revenue = float(results["What is total revenue?"]["total_revenue_usd"].iloc[0])
overall_roas = float(results["What is overall ROAS?"]["overall_roas"].iloc[0])

by_place = results["Revenue by placement"].copy()
top_place = by_place.iloc[0]["placement"] if len(by_place) else None

top_adv = results["Top 10 advertisers by revenue"].iloc[0]["advertiser_name"]
top_adv_rev = float(results["Top 10 advertisers by revenue"].iloc[0]["revenue"])

top_prod = results["Top 5 products by ROAS last 60 days"].iloc[0]["product_name"]
top_prod_roas = float(results["Top 5 products by ROAS last 60 days"].iloc[0]["roas"])

summary = (
    f"Across all transactions in the dataset, total revenue is about ${total_revenue:,.0f} and overall ROAS is ~{overall_roas:.2f}. "
    + (f"The highest-revenue placement is {top_place}. " if top_place is not None else "")
    + f"The top advertiser by revenue in this run is {top_adv} (~${top_adv_rev:,.0f}). "
    + f"Over the last 60 days, the top product by ROAS is {top_prod} (~{top_prod_roas:.2f} ROAS)."
)

print(summary)


Across all transactions in the dataset, total revenue is about $3,983,590 and overall ROAS is ~1.51. The highest-revenue placement is Feed. The top advertiser by revenue in this run is Nova Commerce (~$319,616). Over the last 60 days, the top product by ROAS is App Install Ads (~1.86 ROAS).


In [8]:
conn.close()
